Structured Output

Models can be requested to provide their response in format matching a given schema . This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforing structured output.


Pydantic 

pydantic models provide the richest feature set with Field validation , description and nested structures.

In [7]:
import os 
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b") # reasoning model 
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000013E4BB87C50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013E4CD2C690>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="the title of the movie")
    year:int = Field(description="this year the movie was released")
    director:str = Field(description="the director of the movie")
    rating:float = Field(description="the movie rating out of 10")

In [13]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000013E4BB87C50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013E4CD2C690>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'the title of the movie', 'type': 'string'}, 'year': {'description': 'this year the movie was released', 'type': 'integer'}, 'director': {'description': 'the director of the movie', 'type': 'string'}, 'rating': {'description': 'the movie rating out 

In [15]:
model.invoke("Provide me the deatils of movie slum dog millionare")

AIMessage(content='<think>\nOkay, I need to provide details about the movie "Slumdog Millionaire." Let me start by recalling what I know. The movie is a British drama, I think. It\'s based on a novel, maybe "Q&A" by Vikas Swarup. The main character is Jamal Malik, who\'s on the Indian version of Who Wants to Be a Millionaire? The film\'s director is Danny Boyle, right? The cast includes Dev Patel as the young Jamal, and Freida Pinto as Latika. There\'s also a story about how Jamal uses his past experiences to answer the questions in the game show. The movie won several Oscars. Let me check the release year—2008, I believe. The setting is Mumbai, with a lot of scenes in the slums. The music is by A.R. Rahman, which was nominated for several awards. The film\'s structure alternates between the game show and Jamal\'s flashbacks. The themes include love, poverty, and fate. It\'s a mix of drama and thriller elements. The ending is bittersweet, with Jamal finding Latika and being arrested, b

In [16]:
response = model_with_structure.invoke("Provide details about the movie slum Dog millionare")
response 

Movie(title='Slum Dog Millionaire', year=2008, director='Danny Boyle', rating=8.0)

Message output alongside parsed structure

In [19]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(...,description="the title of the movie")
    year:int = Field(...,description="this year the movie was released")
    director:str = Field(...,description="the director of the movie")
    rating:float = Field(...,description="the movie rating out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Shershah")
response 

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Shershah. Let me check what I need to do. The available tool is the Movie function, which requires title, year, director, and rating. I need to call this function with the correct parameters.\n\nFirst, I should confirm the title. Shershah is a 2021 Indian biographical action film. The director is Vishnu Varadhan. The release year is 2021. As for the rating, I think it's around 8.2 on IMDb. Let me make sure those details are accurate. Once I have all the required parameters, I can structure the tool call accordingly. I need to ensure all required fields are included: title, year, director, and rating. Let me double-check each piece of information to avoid any mistakes.\n", 'tool_calls': [{'id': 'm9ae5zntx', 'function': {'arguments': '{"director":"Vishnu Varadhan","rating":8.2,"title":"Shershah","year":2021}', 'name': 'Movie'}, 'type': 'function'}]}, response_metada

## nested structure

In [22]:
from pydantic import BaseModel , Field

class Actor(BaseModel):
    name:str 
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget : float | None = Field(None,description="Budget in miilions USD ")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("provide the deatils of movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Jacob'), Actor(name='Josh Holloway', role='Eames')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

TypedDict


TypedDict provides a similar alternative using python built-in typing , ideal when you don't need runtime validation. 

In [24]:
from typing_extensions import TypedDict , Annotated

class MovieDict(TypedDict):
    """ A movie with details."""
    title: Annotated[str,...,"title of the movie"]
    year: Annotated[int,...,"the year of the movie was released"]
    director: Annotated[str,...,"the director of the movie"]
    rating: Annotated[str,...,"the movie's rating out of 10"]

model.with_structured_output(MovieDict)

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the deatils of the movie Batman")
response

{'director': 'Tim Burton', 'rating': '8.0', 'title': 'Batman', 'year': 1989}

In [25]:

class Actor(TypedDict):
    name:str 
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget : float | None = Field(None,description="Budget in miilions USD ")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("provide the deatils of movie Inception")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'James'}],
 'genres': ['Action', 'Science Fiction', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [30]:
# model_withtypedict.profile # when we create type dict or structured output no profile 
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [38]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain_groq import ChatGroq

class ContactInfo(BaseModel):
    """ Contact information for a person """
    name:str = Field(description="the name of the person")
    email:str = Field(description="the email of the address of the person")
    phone:str = Field(description="the phone numeber of the person")

model = init_chat_model("groq:qwen/qwen3-32b")
agent = create_agent(
    model = model,
    response_format = ContactInfo  # autoselcts provider strategy
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from : Bharat Singh , Bharat@gmail.com , (91) 9999444455 "}]
})
result

{'messages': [HumanMessage(content='Extract contact info from : Bharat Singh , Bharat@gmail.com , (91) 9999444455 ', additional_kwargs={}, response_metadata={}, id='2c968805-2be0-41a3-b2ce-0deb42b0e846'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text: "Bharat Singh , Bharat@gmail.com , (91) 9999444455". \n\nFirst, I need to identify the different parts. The name is probably "Bharat Singh" since that\'s a common name format. Then the email looks like "Bharat@gmail.com". The phone number is "(91) 9999444455". \n\nWait, the phone number has a country code 91, which is India\'s code. The format with parentheses around the country code is a bit unusual. Maybe I should present it as is, but sometimes people write phone numbers without the parentheses. However, the user provided it with the parentheses, so I should include them as part of the phone number.\n\nNow, checking the required fi

## using TypeDict

In [39]:
from typing_extensions import TypedDict 
from langchain.agents import create_agent
from langchain_groq import ChatGroq

class ContactInfo(TypedDict):
    """ Contact information for a person """
    name:str 
    email:str  
    phone:str  

model = init_chat_model("groq:qwen/qwen3-32b")
agent = create_agent(
    model = model,
    response_format = ContactInfo  # autoselcts provider strategy
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from : Bharat Singh , Bharat@gmail.com , (91) 9999444455 "}]
})
result

{'messages': [HumanMessage(content='Extract contact info from : Bharat Singh , Bharat@gmail.com , (91) 9999444455 ', additional_kwargs={}, response_metadata={}, id='4c517fb1-4313-48cb-8dd8-301f22a5ecef'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text. The input is "Bharat Singh , Bharat@gmail.com , (91) 9999444455". \n\nFirst, I need to identify the different parts of the contact info. The name is probably "Bharat Singh" since that\'s the first part before the comma. Then there\'s an email address: "Bharat@gmail.com". The phone number is "(91) 9999444455". \n\nWait, the phone number has a country code (91) which is India\'s code. The format here is (91) followed by the number. The function parameters require name, email, and phone. So I need to structure these into the ContactInfo function.\n\nI should check if all required fields are present. Name, email, and phone are all there. 

In [40]:
from typing_extensions import TypedDict 
from langchain.agents import create_agent
from langchain_groq import ChatGroq

class ContactInfo(TypedDict):
    """ Contact information for a person """
    name:str 
    email:str  
    phone:str  

model = init_chat_model("groq:qwen/qwen3-32b")
agent = create_agent(
    model = model,
    response_format = ContactInfo  # autoselcts provider strategy
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from : Bharat Singh , Bharat@gmail.com , (91) 9999444455 "}]
})
result
result["structured_response"]

{'name': 'Bharat Singh',
 'email': 'Bharat@gmail.com',
 'phone': '(91) 9999444455'}

## data classes

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_groq import ChatGroq

@dataclass
class ContactInfo:
    """ Contact information for a person """
    name:str 
    email:str  
    phone:str  

model = init_chat_model("groq:qwen/qwen3-32b")
agent = create_agent(
    model = model,
    response_format = ContactInfo  # autoselcts provider strategy
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from : Bharat Singh , Bharat@gmail.com , (91) 9999444455 "}]
})
result